# Emotion Analysis Demo

This notebook demonstrates emotion detection and analysis using the ASDRP library. We'll explore:

1. Detecting emotions from facial landmarks
2. Understanding Action Units (FACS)
3. Visualizing emotion predictions and confidence scores
4. Analyzing emotion probabilities over time
5. Comparing emotions across different frames

## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from asdrp import (
    MediaPipeFaceDetector, VideoFileReader,
    GeometryBasedEmotionAnalyzer, EmotionType,
    TemporalEmotionAnalyzer
)
from asdrp.visualization import (
    plot_emotion_distribution,
    plot_emotion_timeline,
    plot_confidence_over_time,
    plot_emotion_probabilities_over_time,
    EmotionDisplay
)

# Configure plotting
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid")

print("Setup complete!")

## 1. Initialize Components

Let's set up the face detector and emotion analyzer.

In [ ]:
# Paths
video_path = project_root / "data" / "videos" / "youtube_short_emotion.mp4"
model_path = project_root / "models" / "face_landmarker.task"

# Check files
if not video_path.exists():
    print(f"Error: Video not found at {video_path}")
if not model_path.exists():
    print(f"Error: Model not found at {model_path}")
    print("Please download from: https://developers.google.com/mediapipe/solutions/vision/face_landmarker")
else:
    print(f"✓ Video: {video_path}")
    print(f"✓ Model: {model_path}")

In [ ]:
# Initialize face detector
if model_path.exists():
    face_detector = MediaPipeFaceDetector(
        model_path=str(model_path),
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        num_faces=1,
        running_mode="VIDEO"
    )
    print("✓ Face detector initialized")
    
    # Initialize emotion analyzer
    emotion_analyzer = GeometryBasedEmotionAnalyzer(
        confidence_threshold=0.3
    )
    print("✓ Emotion analyzer initialized")
    
    # Initialize temporal analyzer for smoothing
    temporal_analyzer = TemporalEmotionAnalyzer(
        window_size=5,
        min_confidence=0.3
    )
    print("✓ Temporal analyzer initialized")

## 2. Process Video and Detect Emotions

Let's process the video and detect emotions for each frame.

In [ ]:
# Process video frames
if model_path.exists():
    with VideoFileReader(str(video_path)) as reader:
        metadata = reader.metadata
        print(f"Processing video: {metadata.width}x{metadata.height} @ {metadata.fps:.2f} FPS")
        print(f"Total frames: {metadata.total_frames}")
        
        # Sample frames (process every Nth frame for faster analysis)
        skip_frames = 2  # Process every 3rd frame
        frames_to_process = list(range(0, min(metadata.total_frames, 300), skip_frames + 1))
        
        results = []
        print(f"\nProcessing {len(frames_to_process)} frames...")
        
        for i, frame_num in enumerate(frames_to_process):
            frame_data = reader.get_frame_at(frame_num)
            if frame_data is None:
                continue
            
            # Detect face
            faces = face_detector.detect(frame_data.frame, timestamp_ms=frame_data.timestamp_ms)
            
            if faces:
                face = faces[0]
                
                # Analyze emotion
                emotion_prediction = emotion_analyzer.analyze(face)
                
                # Apply temporal smoothing
                smoothed_prediction = temporal_analyzer.smooth_prediction(emotion_prediction)
                
                results.append({
                    'frame_number': frame_num,
                    'timestamp_ms': frame_data.timestamp_ms,
                    'frame': frame_data.frame,
                    'face': face,
                    'prediction': emotion_prediction,
                    'smoothed': smoothed_prediction
                })
            
            # Progress
            if (i + 1) % 20 == 0:
                print(f"  Processed {i + 1}/{len(frames_to_process)} frames...")
        
        print(f"\n✓ Completed! Analyzed {len(results)} frames with face detections")

## 3. Visualize Sample Emotion Detections

Let's display frames with detected emotions.

In [ ]:
# Display sample frames with emotions
if results:
    # Select diverse samples
    num_samples = 6
    sample_indices = np.linspace(0, len(results) - 1, num_samples, dtype=int)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    emotion_display = EmotionDisplay()
    
    for idx, result_idx in enumerate(sample_indices):
        result = results[result_idx]
        
        # Annotate frame with emotion
        annotated = emotion_display.draw_emotion_label(
            result['frame'].copy(),
            result['smoothed'],
            position=(50, 50),
            show_confidence=True,
            show_probabilities=True
        )
        
        # Display
        axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(
            f"Frame {result['frame_number']} @ {result['timestamp_ms']/1000:.2f}s\n"
            f"{result['smoothed'].emotion.value.upper()} "
            f"({result['smoothed'].confidence:.2f})",
            fontsize=11, fontweight='bold'
        )
        axes[idx].axis('off')
    
    plt.suptitle("Emotion Detection Results", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Action Unit Analysis

Action Units (AUs) are the building blocks of facial expressions based on the Facial Action Coding System (FACS). Let's examine which AUs are detected.

In [ ]:
# Analyze Action Units
if results:
    # Collect all detected AUs
    au_detections = defaultdict(list)
    
    for result in results:
        prediction = result['prediction']
        for au in prediction.action_units:
            if au.present:
                au_detections[au.au_type].append({
                    'frame': result['frame_number'],
                    'intensity': au.intensity,
                    'confidence': au.confidence,
                    'emotion': prediction.emotion
                })
    
    # Create summary
    print("Action Unit Detection Summary:")
    print("=" * 70)
    
    if au_detections:
        for au_type in sorted(au_detections.keys(), key=lambda x: x.value):
            detections = au_detections[au_type]
            avg_intensity = np.mean([d['intensity'] for d in detections])
            detection_count = len(detections)
            detection_rate = (detection_count / len(results)) * 100
            
            print(f"\n{au_type} (AU{au_type.value}):")
            print(f"  Detected in: {detection_count}/{len(results)} frames ({detection_rate:.1f}%)")
            print(f"  Avg intensity: {avg_intensity:.3f}")
            print(f"  Associated emotions: {set(d['emotion'].value for d in detections[:5])}")
    else:
        print("No action units detected with sufficient confidence.")

In [ ]:
# Visualize AU intensity over time
if results and au_detections:
    # Select top AUs by frequency
    top_aus = sorted(au_detections.items(), 
                     key=lambda x: len(x[1]), 
                     reverse=True)[:6]
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, (au_type, detections) in enumerate(top_aus):
        if idx >= 6:
            break
        
        # Extract data
        frames = [d['frame'] for d in detections]
        intensities = [d['intensity'] for d in detections]
        
        # Plot
        axes[idx].scatter(frames, intensities, alpha=0.6, s=30)
        axes[idx].plot(frames, intensities, alpha=0.3, linewidth=1)
        axes[idx].set_xlabel('Frame Number', fontsize=10)
        axes[idx].set_ylabel('Intensity', fontsize=10)
        axes[idx].set_title(f'{au_type} Intensity Over Time', fontsize=11, fontweight='bold')
        axes[idx].set_ylim(0, 1)
        axes[idx].grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(top_aus), 6):
        axes[idx].axis('off')
    
    plt.suptitle('Action Unit Intensities Across Video', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Emotion Distribution Analysis

Let's analyze the distribution of emotions across all frames.

In [ ]:
# Emotion distribution
if results:
    predictions = [r['smoothed'] for r in results]
    
    # Count emotions
    emotion_counts = defaultdict(int)
    for pred in predictions:
        emotion_counts[pred.emotion] += 1
    
    # Create bar plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    emotions = sorted(emotion_counts.keys(), key=lambda x: emotion_counts[x], reverse=True)
    counts = [emotion_counts[e] for e in emotions]
    labels = [e.value.capitalize() for e in emotions]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE']
    
    bars = axes[0].bar(labels, counts, color=colors[:len(labels)], alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Emotion', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Frame Count', fontsize=12, fontweight='bold')
    axes[0].set_title('Emotion Distribution (Bar Chart)', fontsize=14, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Add count labels
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}\n({height/len(results)*100:.1f}%)',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Pie chart
    wedges, texts, autotexts = axes[1].pie(
        counts, labels=labels, autopct='%1.1f%%',
        colors=colors[:len(labels)], startangle=90,
        explode=[0.05 if i == 0 else 0 for i in range(len(counts))]
    )
    
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(11)
        autotext.set_fontweight('bold')
    
    for text in texts:
        text.set_fontsize(11)
        text.set_fontweight('bold')
    
    axes[1].set_title('Emotion Distribution (Pie Chart)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\nEmotion Statistics:")
    print("=" * 60)
    total_frames = len(results)
    for emotion in emotions:
        count = emotion_counts[emotion]
        percentage = (count / total_frames) * 100
        print(f"{emotion.value.capitalize():12s}: {count:3d} frames ({percentage:5.1f}%)")

## 6. Emotion Timeline Visualization

Let's visualize how emotions change over time throughout the video.

In [ ]:
# Emotion timeline
if results:
    # Extract timeline data
    frames = [r['frame_number'] for r in results]
    timestamps = [r['timestamp_ms'] / 1000 for r in results]  # Convert to seconds
    emotions = [r['smoothed'].emotion for r in results]
    confidences = [r['smoothed'].confidence for r in results]
    
    # Map emotions to numeric values for plotting
    emotion_types = list(EmotionType)
    emotion_to_num = {e: i for i, e in enumerate(emotion_types)}
    emotion_values = [emotion_to_num[e] for e in emotions]
    
    # Create timeline plot
    fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
    
    # Emotion timeline
    colors_map = {
        EmotionType.NEUTRAL: '#C0C0C0',
        EmotionType.HAPPY: '#FFD700',
        EmotionType.SAD: '#4682B4',
        EmotionType.ANGRY: '#DC143C',
        EmotionType.SURPRISED: '#FF8C00',
        EmotionType.FEARFUL: '#9370DB',
        EmotionType.DISGUSTED: '#228B22'
    }
    
    for i in range(len(timestamps) - 1):
        color = colors_map.get(emotions[i], '#808080')
        axes[0].plot(timestamps[i:i+2], emotion_values[i:i+2], 
                    color=color, linewidth=3, marker='o', markersize=4)
    
    axes[0].set_ylabel('Emotion', fontsize=12, fontweight='bold')
    axes[0].set_yticks(range(len(emotion_types)))
    axes[0].set_yticklabels([e.value.capitalize() for e in emotion_types], fontsize=10)
    axes[0].set_title('Emotion Timeline', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Confidence timeline
    axes[1].plot(timestamps, confidences, color='#2E86AB', linewidth=2, marker='o', markersize=3)
    axes[1].fill_between(timestamps, confidences, alpha=0.3, color='#2E86AB')
    axes[1].axhline(y=0.5, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Threshold')
    axes[1].set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Confidence', fontsize=12, fontweight='bold')
    axes[1].set_title('Confidence Score Over Time', fontsize=14, fontweight='bold')
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print("\nConfidence Statistics:")
    print("=" * 60)
    print(f"Mean confidence: {np.mean(confidences):.3f}")
    print(f"Std confidence:  {np.std(confidences):.3f}")
    print(f"Min confidence:  {np.min(confidences):.3f}")
    print(f"Max confidence:  {np.max(confidences):.3f}")
    print(f"Median:          {np.median(confidences):.3f}")

## 7. Emotion Probability Analysis

Let's examine the probability distribution for all emotions over time.

In [ ]:
# Emotion probabilities over time
if results:
    # Extract probability data
    timestamps = [r['timestamp_ms'] / 1000 for r in results]
    
    # Organize probabilities by emotion
    prob_data = {emotion: [] for emotion in EmotionType}
    
    for result in results:
        probs = result['smoothed'].probabilities
        for emotion in EmotionType:
            prob_data[emotion].append(probs.get(emotion, 0.0))
    
    # Create stacked area plot
    fig, ax = plt.subplots(figsize=(18, 8))
    
    # Prepare data for stacking
    colors = ['#C0C0C0', '#FFD700', '#4682B4', '#DC143C', '#FF8C00', '#9370DB', '#228B22']
    emotion_labels = [e.value.capitalize() for e in EmotionType]
    
    # Stack probabilities
    prob_matrix = np.array([prob_data[emotion] for emotion in EmotionType])
    
    ax.stackplot(timestamps, prob_matrix, labels=emotion_labels, 
                colors=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Probability', fontsize=12, fontweight='bold')
    ax.set_title('Emotion Probability Distribution Over Time', fontsize=16, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Frame Comparison

Let's compare different emotions detected in the video side by side.

In [ ]:
# Find frames with different emotions
if results:
    # Group frames by emotion
    emotion_frames = defaultdict(list)
    for result in results:
        emotion_frames[result['smoothed'].emotion].append(result)
    
    # Select one frame per detected emotion
    sample_frames = {}
    for emotion, frames in emotion_frames.items():
        # Pick the frame with highest confidence
        best_frame = max(frames, key=lambda x: x['smoothed'].confidence)
        sample_frames[emotion] = best_frame
    
    # Display comparison
    num_emotions = len(sample_frames)
    cols = 3
    rows = (num_emotions + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    emotion_display = EmotionDisplay()
    
    for idx, (emotion, result) in enumerate(sorted(sample_frames.items(), 
                                                   key=lambda x: x[1]['smoothed'].confidence,
                                                   reverse=True)):
        # Annotate frame
        annotated = emotion_display.draw_emotion_label(
            result['frame'].copy(),
            result['smoothed'],
            position=(30, 30),
            show_confidence=True,
            show_probabilities=True
        )
        
        axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(
            f"{emotion.value.upper()}\n"
            f"Frame {result['frame_number']} @ {result['timestamp_ms']/1000:.2f}s\n"
            f"Confidence: {result['smoothed'].confidence:.3f}",
            fontsize=12, fontweight='bold'
        )
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(len(sample_frames), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('Detected Emotions Comparison', fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print details
    print("\nDetected Emotions Summary:")
    print("=" * 70)
    for emotion, result in sorted(sample_frames.items(), key=lambda x: x[0].value):
        print(f"\n{emotion.value.capitalize()}:")
        print(f"  Frame: {result['frame_number']}")
        print(f"  Time: {result['timestamp_ms']/1000:.2f}s")
        print(f"  Confidence: {result['smoothed'].confidence:.3f}")
        print(f"  Top 3 probabilities:")
        top_probs = sorted(result['smoothed'].probabilities.items(), 
                          key=lambda x: x[1], reverse=True)[:3]
        for emo, prob in top_probs:
            print(f"    {emo.value}: {prob:.3f}")

## 9. Heatmap of Emotion Probabilities

Let's create a heatmap showing all emotion probabilities across time.

In [ ]:
# Create emotion probability heatmap
if results:
    # Prepare data matrix
    emotion_types = list(EmotionType)
    prob_matrix = np.zeros((len(emotion_types), len(results)))
    
    for col, result in enumerate(results):
        for row, emotion in enumerate(emotion_types):
            prob_matrix[row, col] = result['smoothed'].probabilities.get(emotion, 0.0)
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(18, 8))
    
    # Use only subset of frames for readability
    step = max(1, len(results) // 50)
    prob_matrix_subset = prob_matrix[:, ::step]
    timestamps_subset = [results[i]['timestamp_ms']/1000 for i in range(0, len(results), step)]
    
    im = ax.imshow(prob_matrix_subset, aspect='auto', cmap='YlOrRd', interpolation='bilinear')
    
    # Set ticks
    ax.set_yticks(range(len(emotion_types)))
    ax.set_yticklabels([e.value.capitalize() for e in emotion_types], fontsize=11)
    
    # X-axis: time points
    num_xticks = 10
    xtick_indices = np.linspace(0, len(timestamps_subset) - 1, num_xticks, dtype=int)
    ax.set_xticks(xtick_indices)
    ax.set_xticklabels([f"{timestamps_subset[i]:.1f}s" for i in xtick_indices], 
                       rotation=45, ha='right', fontsize=10)
    
    ax.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax.set_ylabel('Emotion', fontsize=12, fontweight='bold')
    ax.set_title('Emotion Probability Heatmap Over Time', fontsize=16, fontweight='bold')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Probability', rotation=270, labelpad=20, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## Summary

In this notebook, we explored:

1. Setting up emotion detection with the ASDRP library
2. Processing video frames to detect faces and emotions
3. Understanding Action Units (FACS) and their role in emotion detection
4. Visualizing emotion predictions with confidence scores
5. Analyzing emotion distribution across the video
6. Creating timeline visualizations of emotions
7. Examining emotion probability distributions
8. Comparing different detected emotions
9. Creating heatmaps for temporal emotion analysis

### Key Findings

The geometry-based emotion analyzer uses facial landmarks to:
- Detect Action Units (muscle movements)
- Classify emotions based on AU combinations
- Provide confidence scores for predictions
- Generate probability distributions across all emotion types

### Next Steps

- **03_temporal_analysis.ipynb**: Deep dive into temporal patterns and microexpressions

### Clean Up

In [ ]:
# Clean up
if model_path.exists():
    face_detector.close()
    print("Resources cleaned up successfully!")